# Проверка состоятельности fallback-ключа `INN + номер договора`

Тетрадка оценивает, можно ли использовать `INN + contract` как fallback для построения витрины,
если `agr_id` отсутствует в части данных Озера.


In [ ]:
import re
import pandas as pd
from rail_connectors.connection import connect

pd.options.display.max_columns = None
pd.options.display.width = None


In [ ]:
# Параметры
months = ['2026-01-01', '2026-02-01', '2026-03-01']
acq_class = 'SA'

# QC-пороги (можно откалибровать под DRP SLA)
THRESHOLDS = {
    'max_unresolved_share': 0.05,   # unresolved = (agr_candidates=0 or >1)
    'max_ambiguous_share': 0.01,    # >1 кандидатов agr_id
    'max_agr_switch_share': 0.01    # нестабильные ключи между месяцами
}

print('months =', months)
print('thresholds =', THRESHOLDS)


In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()
with imp:
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
print('Impala initialized')


In [ ]:
def normalize_agr_id(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D', '', str(v).strip())
    return s if s else None

def load_key_map(month_start):
    sql = f"""
    select
        cast('{month_start}' as date) as month_start,
        case
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 10, '0')
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 12, '0')
          else regexp_replace(trim(c.c_inn), '[^0-9]', '')
        end as inn_norm,
        lower(trim(cast(a.c_agr_number as string))) as contract_norm,
        cast(a.abs_agr_id as string) as agr_id_norm,
        cast(a.n_agr as string) as n_agr
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = '{acq_class}'
      and cast(a.d_valid_from as date) <= cast('{month_start}' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
    """
    with imp:
        df = imp.fetch(sql)
    if len(df) == 0:
        return df
    df = df.dropna(subset=['inn_norm', 'contract_norm']).copy()
    df['agr_id_norm'] = df['agr_id_norm'].apply(normalize_agr_id)
    df['month_start'] = pd.to_datetime(df['month_start']).dt.date
    return df


## 1) Качество ключа по месяцам: покрытие, неоднозначность, unresolved


In [ ]:
maps = []
for m in months:
    maps.append(load_key_map(m))

key_map_raw = pd.concat(maps, ignore_index=True) if len(maps) else pd.DataFrame()

key_card = (
    key_map_raw.groupby(['month_start', 'inn_norm', 'contract_norm'], as_index=False)
    .agg(
        lake_rows=('n_agr', 'count'),
        agr_candidates=('agr_id_norm', lambda s: int(s.dropna().nunique())),
        rows_without_agr=('agr_id_norm', lambda s: int(s.isna().sum()))
    )
)

key_card['status'] = key_card['agr_candidates'].apply(
    lambda x: 'matched_unique' if x == 1 else ('no_agr_id' if x == 0 else 'ambiguous')
)

monthly_quality = (
    key_card.groupby('month_start', as_index=False)
    .agg(
        unique_inn_contract=('inn_norm', 'count'),
        keys_1_to_1=('agr_candidates', lambda s: int((s == 1).sum())),
        keys_no_agr_id=('agr_candidates', lambda s: int((s == 0).sum())),
        keys_ambiguous_gt1=('agr_candidates', lambda s: int((s > 1).sum()))
    )
)

monthly_quality['share_1_to_1'] = monthly_quality['keys_1_to_1'] / monthly_quality['unique_inn_contract']
monthly_quality['share_no_agr_id'] = monthly_quality['keys_no_agr_id'] / monthly_quality['unique_inn_contract']
monthly_quality['share_ambiguous'] = monthly_quality['keys_ambiguous_gt1'] / monthly_quality['unique_inn_contract']
monthly_quality['share_unresolved'] = (monthly_quality['keys_no_agr_id'] + monthly_quality['keys_ambiguous_gt1']) / monthly_quality['unique_inn_contract']

display(monthly_quality.sort_values('month_start'))
display(key_card[key_card['status'] != 'matched_unique'].head(200))


## 2) Стабильность fallback-ключа между месяцами (`INN+contract -> agr_id`)


In [ ]:
unique_map = key_card[key_card['agr_candidates'] == 1][['month_start', 'inn_norm', 'contract_norm']].copy()
unique_map = unique_map.merge(
    key_map_raw[['month_start', 'inn_norm', 'contract_norm', 'agr_id_norm']].dropna(subset=['agr_id_norm']).drop_duplicates(),
    on=['month_start', 'inn_norm', 'contract_norm'],
    how='left'
)

stability = (
    unique_map.groupby(['inn_norm', 'contract_norm'], as_index=False)
    .agg(
        months_present=('month_start', 'nunique'),
        agr_id_variants=('agr_id_norm', 'nunique')
    )
)
stability['agr_switch_flag'] = (stability['agr_id_variants'] > 1).astype(int)

stability_summary = pd.DataFrame([{
    'keys_observed_ge1_month': len(stability),
    'keys_with_agr_switch': int(stability['agr_switch_flag'].sum()),
    'agr_switch_share': float(stability['agr_switch_flag'].mean() if len(stability) else 0.0)
}])

display(stability_summary)
display(stability[stability['agr_switch_flag'] == 1].head(200))


## 3) Иерархия ключей для витрины: `tier1(agr_id)` / `tier2(fallback)` / `tier3(unresolved)`


In [ ]:
tier_stats = monthly_quality[['month_start', 'unique_inn_contract', 'keys_1_to_1', 'keys_no_agr_id', 'keys_ambiguous_gt1']].copy()
tier_stats = tier_stats.rename(columns={
    'keys_1_to_1': 'tier2_fallback_unique',
    'keys_no_agr_id': 'tier3_no_agr_id',
    'keys_ambiguous_gt1': 'tier3_ambiguous'
})

tier_stats['tier3_unresolved_total'] = tier_stats['tier3_no_agr_id'] + tier_stats['tier3_ambiguous']
tier_stats['tier2_share'] = tier_stats['tier2_fallback_unique'] / tier_stats['unique_inn_contract']
tier_stats['tier3_share'] = tier_stats['tier3_unresolved_total'] / tier_stats['unique_inn_contract']

display(tier_stats.sort_values('month_start'))


## 4) QC-вердикт по применимости подхода для прод-витрины


In [ ]:
qc = monthly_quality.copy()
qc['rule_unresolved_ok'] = qc['share_unresolved'] <= THRESHOLDS['max_unresolved_share']
qc['rule_ambiguous_ok'] = qc['share_ambiguous'] <= THRESHOLDS['max_ambiguous_share']

agr_switch_share = float(stability_summary.loc[0, 'agr_switch_share']) if len(stability_summary) else 0.0
qc['agr_switch_share_global'] = agr_switch_share
qc['rule_stability_ok'] = qc['agr_switch_share_global'] <= THRESHOLDS['max_agr_switch_share']

qc['fallback_viable'] = qc['rule_unresolved_ok'] & qc['rule_ambiguous_ok'] & qc['rule_stability_ok']

display(qc[['month_start', 'share_1_to_1', 'share_unresolved', 'share_ambiguous', 'agr_switch_share_global', 'fallback_viable']])

if qc['fallback_viable'].all():
    print('VERDICT: fallback INN+contract допустим для витрины при текущих порогах.')
else:
    print('VERDICT: fallback INN+contract использовать только с quality_flag и NULL для unresolved.')


## 5) Рекомендованные поля качества в итоговой витрине

Добавить в финальную витрину:
- `key_tier` (`tier1_agr_id` / `tier2_inn_contract_unique` / `tier3_unresolved`)
- `match_status` (`matched_unique` / `no_agr_id` / `ambiguous`)
- `agr_candidates`
- `rows_without_agr`
- `is_fallback_applied` (0/1)
- `dq_block_reason` (например `NO_AGR_ID`, `AMBIGUOUS_KEY`)


## 6) A/B валидация на фактах: `tier1(by_agr_id)` vs `tier2(by_inn_contract)`

Цель:
- проверить, что fallback-ключ корректен не только по структуре ключа, но и по фактическим метрикам;
- сравнить `trx_sum`, `term_cnt`, `retl_cnt` на пересечении ключей, где доступны оба подхода.

In [ ]:
sql_monthly_facts = f"""
with month_dim as (
    select cast(mm as date) as month_start,
           cast(last_day(cast(mm as date)) as date) as month_end
    from (
        select '{months[0]}' as mm
        union all select '{months[1]}'
        union all select '{months[2]}'
    ) s
),
agr_base as (
    select
        md.month_start,
        md.month_end,
        case
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 9 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 10, '0')
          when length(regexp_replace(trim(c.c_inn), '[^0-9]', '')) = 11 then lpad(regexp_replace(trim(c.c_inn), '[^0-9]', ''), 12, '0')
          else regexp_replace(trim(c.c_inn), '[^0-9]', '')
        end as inn_norm,
        lower(trim(cast(a.c_agr_number as string))) as contract_norm,
        cast(a.abs_agr_id as string) as agr_id_norm,
        cast(a.n_agr as string) as n_agr,
        a.n_cmp_client
    from month_dim md
    join ods_alpha.scd1_agreements a
      on cast(a.d_valid_from as date) <= md.month_start
     and (a.d_valid_to is null or cast(a.d_valid_to as date) >= md.month_start)
    join ods_alpha.scd1_companies c
      on c.n_cmp = a.n_cmp_client
    where a.acq_class = '{acq_class}'
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
),
trx_f as (
    select
        cast(trunc(cast(t.d_trans_date as date), 'MM') as date) as month_start,
        t.n_trx,
        cast(t.n_amt_src as double) as n_amt_src,
        cast(t.c_nter as string) as c_nter,
        cast(t.c_nmrc as string) as c_nmrc
    from ods_alpha.scd1_trx t
    where cast(t.d_trans_date as date) >= cast('{months[0]}' as date)
      and cast(t.d_trans_date as date) <= cast(last_day(cast('{months[2]}' as date)) as date)
      and t.c_nter is not null
),
trx_by_agr as (
    select
        cast(ta.n_agr as string) as n_agr,
        tf.month_start,
        sum(tf.n_amt_src) as trx_sum
    from trx_f tf
    join ods_alpha.scd1_trx_acq ta
      on ta.n_trx = tf.n_trx
    group by cast(ta.n_agr as string), tf.month_start
),
term_by_cmp as (
    select
        ab.month_start,
        ab.n_cmp_client,
        count(distinct cast(t.c_nter as string)) as term_cnt
    from agr_base ab
    join ods_alpha.scd1_merchants m
      on m.n_cmp = ab.n_cmp_client
     and m.c_nmrc is not null
    join ods_alpha.scd1_pos_terminals t
      on t.c_nmrc = m.c_nmrc
     and t.c_nter is not null
    group by ab.month_start, ab.n_cmp_client
),
retl_by_cmp as (
    select
        ab.month_start,
        ab.n_cmp_client,
        count(distinct cast(m.c_nmrc as string)) as retl_cnt
    from agr_base ab
    join ods_alpha.scd1_merchants m
      on m.n_cmp = ab.n_cmp_client
     and m.c_nmrc is not null
    group by ab.month_start, ab.n_cmp_client
)
select
    ab.month_start,
    ab.inn_norm,
    ab.contract_norm,
    ab.agr_id_norm,
    ab.n_agr,
    coalesce(tba.trx_sum, 0.0) as trx_sum,
    coalesce(tc.term_cnt, 0) as term_cnt,
    coalesce(rc.retl_cnt, 0) as retl_cnt
from agr_base ab
left join trx_by_agr tba
  on tba.n_agr = ab.n_agr
 and tba.month_start = ab.month_start
left join term_by_cmp tc
  on tc.n_cmp_client = ab.n_cmp_client
 and tc.month_start = ab.month_start
left join retl_by_cmp rc
  on rc.n_cmp_client = ab.n_cmp_client
 and rc.month_start = ab.month_start
"""

with imp:
    monthly_facts = imp.fetch(sql_monthly_facts)

monthly_facts['month_start'] = pd.to_datetime(monthly_facts['month_start']).dt.date
monthly_facts['agr_id_norm'] = monthly_facts['agr_id_norm'].apply(normalize_agr_id)

display(monthly_facts.head(50))
print('rows =', len(monthly_facts))

In [ ]:
# Карта fallback-ключа (только уникальные соответствия)
key_unique = key_card[key_card['agr_candidates'] == 1][['month_start', 'inn_norm', 'contract_norm']].copy()
key_unique = key_unique.merge(
    key_map_raw[['month_start', 'inn_norm', 'contract_norm', 'agr_id_norm']].dropna(subset=['agr_id_norm']).drop_duplicates(),
    on=['month_start', 'inn_norm', 'contract_norm'],
    how='left'
).rename(columns={'agr_id_norm': 'agr_id_from_fallback'})

# tier1 факт (по agr_id из строки)
tier1 = monthly_facts.dropna(subset=['agr_id_norm']).copy()
tier1 = tier1.rename(columns={'agr_id_norm': 'agr_id_from_row'})

# tier2 факт (через fallback INN+contract)
tier2 = (
    monthly_facts[['month_start', 'inn_norm', 'contract_norm']].drop_duplicates()
    .merge(key_unique, on=['month_start', 'inn_norm', 'contract_norm'], how='inner')
)

# На тех же ключах подтягиваем факты по найденному agr_id
facts_by_agr = monthly_facts[['month_start', 'agr_id_norm', 'trx_sum', 'term_cnt', 'retl_cnt']].drop_duplicates()
tier2 = tier2.merge(
    facts_by_agr,
    left_on=['month_start', 'agr_id_from_fallback'],
    right_on=['month_start', 'agr_id_norm'],
    how='left'
)
tier2 = tier2.drop(columns=['agr_id_norm'])

# A/B на пересечении: есть и agr_id строки, и fallback
ab = tier1.merge(
    tier2,
    left_on=['month_start', 'inn_norm', 'contract_norm'],
    right_on=['month_start', 'inn_norm', 'contract_norm'],
    how='inner',
    suffixes=('_tier1', '_tier2')
)

ab['same_agr'] = (ab['agr_id_from_row'] == ab['agr_id_from_fallback'])
ab['diff_trx_sum'] = ab['trx_sum_tier2'] - ab['trx_sum_tier1']
ab['diff_term_cnt'] = ab['term_cnt_tier2'] - ab['term_cnt_tier1']
ab['diff_retl_cnt'] = ab['retl_cnt_tier2'] - ab['retl_cnt_tier1']

ab['match_trx_sum'] = ab['diff_trx_sum'].abs() <= 0.01
ab['match_term_cnt'] = ab['diff_term_cnt'] == 0
ab['match_retl_cnt'] = ab['diff_retl_cnt'] == 0
ab['match_all'] = ab['match_trx_sum'] & ab['match_term_cnt'] & ab['match_retl_cnt']

display(ab.head(100))
print('ab_rows =', len(ab))

In [ ]:
ab_summary = pd.DataFrame([
    {'metric': 'ab_rows_total', 'value': len(ab)},
    {'metric': 'ab_same_agr_rows', 'value': int(ab['same_agr'].sum()) if len(ab) else 0},
    {'metric': 'ab_same_agr_share', 'value': float(ab['same_agr'].mean()) if len(ab) else 0.0},
    {'metric': 'ab_match_trx_sum_rows', 'value': int(ab['match_trx_sum'].sum()) if len(ab) else 0},
    {'metric': 'ab_match_term_cnt_rows', 'value': int(ab['match_term_cnt'].sum()) if len(ab) else 0},
    {'metric': 'ab_match_retl_cnt_rows', 'value': int(ab['match_retl_cnt'].sum()) if len(ab) else 0},
    {'metric': 'ab_match_all_rows', 'value': int(ab['match_all'].sum()) if len(ab) else 0},
    {'metric': 'ab_mismatch_any_rows', 'value': int((~ab['match_all']).sum()) if len(ab) else 0},
])

print('A/B summary (tier1 vs tier2):')
display(ab_summary)

print('Mismatches (top 200):')
display(
    ab[(~ab['match_all']) | (~ab['same_agr'])][[
        'month_start', 'inn_norm', 'contract_norm',
        'agr_id_from_row', 'agr_id_from_fallback', 'same_agr',
        'trx_sum_tier1', 'trx_sum_tier2', 'diff_trx_sum',
        'term_cnt_tier1', 'term_cnt_tier2', 'diff_term_cnt',
        'retl_cnt_tier1', 'retl_cnt_tier2', 'diff_retl_cnt'
    ]].head(200)
)

### Интерпретация блока A/B

- Если `ab_same_agr_share` близко к 1.0 и `ab_match_all_rows / ab_rows_total` высокое,
  fallback-ключ `INN+contract` можно применять шире для витрины.
- Если есть `same_agr = 0`, нужно разбирать нормализацию договора и темпоральную валидность.
- Если `same_agr = 1`, но есть расхождения в метриках, причина обычно в разных периметрах расчета
  (особенно для `term_cnt`/`retl_cnt` при company-based агрегации).
- Для прод-загрузки блокирующее правило: строки с `tier3_unresolved` не обогащать fallback-метриками,
  только `NULL + dq_block_reason`.

## 7) Production key-policy (tier1/tier2/tier3) и блокировка unresolved

В этом блоке формируется единая таблица разрешения ключа для витрины:
- `tier1_agr_id` — если `agr_id` есть в записи;
- `tier2_inn_contract_unique` — если `agr_id` отсутствует, но `INN+contract` дает ровно 1 кандидат;
- `tier3_unresolved` — если `INN+contract` не дает кандидата или дает >1.

Для `tier3_unresolved` целевое правило: KPI, зависящие от договорного ключа, не заполняются (`NULL`) и выставляется `dq_block_reason`.

In [ ]:
# Нормализованная карта fallback по ключу INN+contract
map_resolution = key_card[['month_start', 'inn_norm', 'contract_norm', 'agr_candidates', 'rows_without_agr']].copy()

map_resolution = map_resolution.merge(
    key_map_raw[['month_start', 'inn_norm', 'contract_norm', 'agr_id_norm']].dropna(subset=['agr_id_norm']).drop_duplicates(),
    on=['month_start', 'inn_norm', 'contract_norm'],
    how='left'
)

# Если agr_candidates != 1, agr_id_fallback должен быть NULL
map_resolution['agr_id_fallback'] = map_resolution.apply(
    lambda r: r['agr_id_norm'] if r['agr_candidates'] == 1 else None,
    axis=1
)
map_resolution = map_resolution.drop(columns=['agr_id_norm'])

# Базовая витринная гранулярность (месяц + INN + договор)
base_dataset = monthly_facts[['month_start', 'inn_norm', 'contract_norm', 'agr_id_norm', 'trx_sum', 'term_cnt', 'retl_cnt']].drop_duplicates().copy()
base_dataset = base_dataset.rename(columns={'agr_id_norm': 'agr_id_row'})

base_dataset = base_dataset.merge(
    map_resolution,
    on=['month_start', 'inn_norm', 'contract_norm'],
    how='left'
)

base_dataset['has_agr_row'] = base_dataset['agr_id_row'].notna().astype(int)

base_dataset['resolved_agr_id'] = base_dataset.apply(
    lambda r: r['agr_id_row'] if pd.notna(r['agr_id_row']) else (r['agr_id_fallback'] if r['agr_candidates'] == 1 else None),
    axis=1
)

base_dataset['key_tier'] = base_dataset.apply(
    lambda r: 'tier1_agr_id' if pd.notna(r['agr_id_row']) else ('tier2_inn_contract_unique' if r['agr_candidates'] == 1 else 'tier3_unresolved'),
    axis=1
)

base_dataset['match_status'] = base_dataset['agr_candidates'].apply(
    lambda x: 'matched_unique' if x == 1 else ('no_agr_id' if x == 0 else 'ambiguous')
)

base_dataset['is_fallback_applied'] = ((base_dataset['key_tier'] == 'tier2_inn_contract_unique').astype(int))
base_dataset['dq_block_reason'] = base_dataset['match_status'].map({
    'no_agr_id': 'NO_AGR_ID',
    'ambiguous': 'AMBIGUOUS_KEY'
})
base_dataset.loc[base_dataset['key_tier'] != 'tier3_unresolved', 'dq_block_reason'] = None

display(base_dataset.head(100))
print('base_dataset_rows =', len(base_dataset))

In [ ]:
# Помесячные A/B метрики качества fallback на пересечении tier1/tier2
ab_monthly = (
    ab.groupby('month_start', as_index=False)
    .agg(
        ab_rows=('inn_norm', 'count'),
        same_agr_rows=('same_agr', 'sum'),
        match_trx_sum_rows=('match_trx_sum', 'sum'),
        match_term_cnt_rows=('match_term_cnt', 'sum'),
        match_retl_cnt_rows=('match_retl_cnt', 'sum'),
        match_all_rows=('match_all', 'sum')
    )
)

for c in ['same_agr_rows', 'match_trx_sum_rows', 'match_term_cnt_rows', 'match_retl_cnt_rows', 'match_all_rows']:
    ab_monthly[c] = ab_monthly[c].astype(int)

ab_monthly['same_agr_rate'] = ab_monthly['same_agr_rows'] / ab_monthly['ab_rows']
ab_monthly['match_trx_sum_rate'] = ab_monthly['match_trx_sum_rows'] / ab_monthly['ab_rows']
ab_monthly['match_term_cnt_rate'] = ab_monthly['match_term_cnt_rows'] / ab_monthly['ab_rows']
ab_monthly['match_retl_cnt_rate'] = ab_monthly['match_retl_cnt_rows'] / ab_monthly['ab_rows']
ab_monthly['ab_match_rate'] = ab_monthly['match_all_rows'] / ab_monthly['ab_rows']

display(ab_monthly.sort_values('month_start'))

In [ ]:
# Топ причин расхождений A/B
ab_issues = ab.copy()
ab_issues['issue_type'] = ab_issues.apply(
    lambda r: 'AGR_MISMATCH' if not r['same_agr'] else (
        'METRIC_MISMATCH_TRX' if not r['match_trx_sum'] else (
            'METRIC_MISMATCH_TERM' if not r['match_term_cnt'] else (
                'METRIC_MISMATCH_RETL' if not r['match_retl_cnt'] else 'OK'
            )
        )
    ),
    axis=1
)

issue_summary = (
    ab_issues.groupby(['month_start', 'issue_type'], as_index=False)
    .agg(rows=('inn_norm', 'count'))
    .sort_values(['month_start', 'rows'], ascending=[True, False])
)

display(issue_summary)
display(ab_issues[ab_issues['issue_type'] != 'OK'][[
    'month_start', 'inn_norm', 'contract_norm', 'issue_type',
    'agr_id_from_row', 'agr_id_from_fallback',
    'diff_trx_sum', 'diff_term_cnt', 'diff_retl_cnt'
]].head(200))

## 8) QC-gates для публикации витрины без `agr_id`

Gate-логика:
- `share_unresolved` <= `max_unresolved_share`;
- `share_ambiguous` <= `max_ambiguous_share`;
- `share_key_switch` <= `max_agr_switch_share`;
- `ab_match_rate` >= `min_ab_match_rate`.

Итоговый `publish_gate_pass` считается по каждому месяцу.

In [ ]:
# Дополнительный порог для A/B
THRESHOLDS['min_ab_match_rate'] = 0.95

qc_gates = monthly_quality[['month_start', 'share_unresolved', 'share_ambiguous']].copy()
qc_gates = qc_gates.merge(
    ab_monthly[['month_start', 'ab_match_rate']],
    on='month_start',
    how='left'
)

qc_gates['share_key_switch'] = float(stability_summary.loc[0, 'agr_switch_share']) if len(stability_summary) else 0.0

qc_gates['gate_unresolved_ok'] = qc_gates['share_unresolved'] <= THRESHOLDS['max_unresolved_share']
qc_gates['gate_ambiguous_ok'] = qc_gates['share_ambiguous'] <= THRESHOLDS['max_ambiguous_share']
qc_gates['gate_key_switch_ok'] = qc_gates['share_key_switch'] <= THRESHOLDS['max_agr_switch_share']
qc_gates['gate_ab_match_ok'] = qc_gates['ab_match_rate'].fillna(0.0) >= THRESHOLDS['min_ab_match_rate']

qc_gates['publish_gate_pass'] = (
    qc_gates['gate_unresolved_ok']
    & qc_gates['gate_ambiguous_ok']
    & qc_gates['gate_key_switch_ok']
    & qc_gates['gate_ab_match_ok']
)

qc_gate_summary = pd.DataFrame([
    {'metric': 'months_total', 'value': len(qc_gates)},
    {'metric': 'months_gate_pass', 'value': int(qc_gates['publish_gate_pass'].sum())},
    {'metric': 'months_gate_fail', 'value': int((~qc_gates['publish_gate_pass']).sum())},
    {'metric': 'min_ab_match_rate_threshold', 'value': THRESHOLDS['min_ab_match_rate']},
])

display(qc_gates.sort_values('month_start'))
display(qc_gate_summary)

## 9) Quality-aware dataset для BI (`kpi_total` и `kpi_trusted_only`)

Этот блок формирует dataset с quality-флагами и двумя контурами KPI:
- `kpi_*_total` — как есть по полной выборке;
- `kpi_*_trusted_only` — только для строк с `key_tier != tier3_unresolved`.

In [ ]:
quality_dataset = base_dataset.copy()

quality_dataset['is_trusted'] = (quality_dataset['key_tier'] != 'tier3_unresolved').astype(int)

# KPI в двух контурах
quality_dataset['kpi_trx_sum_total'] = quality_dataset['trx_sum']
quality_dataset['kpi_term_cnt_total'] = quality_dataset['term_cnt']
quality_dataset['kpi_retl_cnt_total'] = quality_dataset['retl_cnt']

quality_dataset['kpi_trx_sum_trusted_only'] = quality_dataset.apply(
    lambda r: r['trx_sum'] if r['is_trusted'] == 1 else None,
    axis=1
)
quality_dataset['kpi_term_cnt_trusted_only'] = quality_dataset.apply(
    lambda r: r['term_cnt'] if r['is_trusted'] == 1 else None,
    axis=1
)
quality_dataset['kpi_retl_cnt_trusted_only'] = quality_dataset.apply(
    lambda r: r['retl_cnt'] if r['is_trusted'] == 1 else None,
    axis=1
)

quality_dataset_monthly = (
    quality_dataset.groupby('month_start', as_index=False)
    .agg(
        rows_cnt=('inn_norm', 'count'),
        trusted_rows=('is_trusted', 'sum'),
        fallback_rows=('is_fallback_applied', 'sum'),
        unresolved_rows=('key_tier', lambda s: int((s == 'tier3_unresolved').sum())),
        kpi_trx_sum_total=('kpi_trx_sum_total', 'sum'),
        kpi_trx_sum_trusted_only=('kpi_trx_sum_trusted_only', 'sum'),
        kpi_term_cnt_total=('kpi_term_cnt_total', 'sum'),
        kpi_term_cnt_trusted_only=('kpi_term_cnt_trusted_only', 'sum'),
        kpi_retl_cnt_total=('kpi_retl_cnt_total', 'sum'),
        kpi_retl_cnt_trusted_only=('kpi_retl_cnt_trusted_only', 'sum')
    )
)

quality_dataset_monthly['trusted_share'] = quality_dataset_monthly['trusted_rows'] / quality_dataset_monthly['rows_cnt']
quality_dataset_monthly['fallback_share'] = quality_dataset_monthly['fallback_rows'] / quality_dataset_monthly['rows_cnt']
quality_dataset_monthly['unresolved_share'] = quality_dataset_monthly['unresolved_rows'] / quality_dataset_monthly['rows_cnt']

display(quality_dataset.head(100))
display(quality_dataset_monthly.sort_values('month_start'))

## 10) Ежемесячный мониторинг и алерты деградации качества ключа

Мониторинг строится на месячных метриках качества и проверяет:
- рост `share_unresolved`;
- падение `ab_match_rate`;
- нарушение QC-gates.

При срабатывании формируется `alerts_table` для операционного контроля.

In [ ]:
monitoring = qc_gates[['month_start', 'share_unresolved', 'share_ambiguous', 'share_key_switch', 'ab_match_rate', 'publish_gate_pass']].copy()
monitoring = monitoring.sort_values('month_start').reset_index(drop=True)

monitoring['mom_unresolved_delta'] = monitoring['share_unresolved'].diff()
monitoring['mom_ab_match_delta'] = monitoring['ab_match_rate'].diff()

# Пороги деградации мониторинга
ALERTS = {
    'max_unresolved_mom_increase': 0.02,  # +2 п.п. месяц-к-месяцу
    'max_ab_match_mom_drop': -0.02        # падение более чем на 2 п.п.
}

monitoring['alert_unresolved_growth'] = monitoring['mom_unresolved_delta'].fillna(0) > ALERTS['max_unresolved_mom_increase']
monitoring['alert_ab_match_drop'] = monitoring['mom_ab_match_delta'].fillna(0) < ALERTS['max_ab_match_mom_drop']
monitoring['alert_gate_fail'] = ~monitoring['publish_gate_pass']

monitoring['alerts_total'] = (
    monitoring['alert_unresolved_growth'].astype(int)
    + monitoring['alert_ab_match_drop'].astype(int)
    + monitoring['alert_gate_fail'].astype(int)
)

alerts_table = monitoring[monitoring['alerts_total'] > 0].copy()

monitoring_summary = pd.DataFrame([
    {'metric': 'months_monitored', 'value': len(monitoring)},
    {'metric': 'months_with_any_alert', 'value': len(alerts_table)},
    {'metric': 'alerts_unresolved_growth', 'value': int(monitoring['alert_unresolved_growth'].sum())},
    {'metric': 'alerts_ab_match_drop', 'value': int(monitoring['alert_ab_match_drop'].sum())},
    {'metric': 'alerts_gate_fail', 'value': int(monitoring['alert_gate_fail'].sum())},
])

display(monitoring)
display(monitoring_summary)
display(alerts_table)